In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

def load_logs(file_path):
    with open(file_path, 'r') as f:
        data = json.load(f)
    return pd.json_normalize(data)

# Load all three runs
spear_df          = load_logs('logs/telemetry_SPEAR_phi3.5-3.8b.json')
spear_hyde_q25_df = load_logs('logs/telemetry_SPEAR_HyDE_qwen2.5-7b.json')
spear_hyde_p35_df = load_logs('logs/telemetry_SPEAR_HyDE_phi3.5-3.8b.json')

# Enforce architecture labels
spear_df['architecture']          = 'SPEAR'
spear_hyde_q25_df['architecture'] = 'SPEAR+HyDE (Qwen2.5:7B)'
spear_hyde_p35_df['architecture'] = 'SPEAR+HyDE (Phi3.5:3.8B)'

# Fixed: Concatenate all THREE
df = pd.concat([spear_df, spear_hyde_q25_df, spear_hyde_p35_df], ignore_index=True)

# Define the order for consistent plotting
arch_order = ['SPEAR', 'SPEAR+HyDE (Qwen2.5:7B)', 'SPEAR+HyDE (Phi3.5:3.8B)']
arch_colors = {
    'SPEAR': '#3498db', 
    'SPEAR+HyDE (Qwen2.5:7B)': '#e74c3c', 
    'SPEAR+HyDE (Phi3.5:3.8B)': '#2ecc71'
}

# Derived columns
critic_cols = ['async_critic_scores.rule_accuracy', 'async_critic_scores.narrative_coherence', 'async_critic_scores.npc_voice_consistency']
df['avg_critic_score'] = df[critic_cols].mean(axis=1)
df['score_per_1k']     = (df['avg_critic_score'] / df['total_tokens']) * 1000

# Fixed: Align phases across the longest run
N_TURNS = max(len(spear_df), len(spear_hyde_q25_df), len(spear_hyde_p35_df))
phase_labels = [
    'Isolation', 'Isolation', 'Isolation',
    'Compound', 'Compound', 'Compound', 'Compound',
    'RAG Edge Cases', 'RAG Edge Cases', 'RAG Edge Cases', 'RAG Edge Cases',
    'RAG Edge Cases', 'RAG Edge Cases', 'RAG Edge Cases', 'RAG Edge Cases',
    'Hallucination Bait', 'Hallucination Bait', 'Hallucination Bait',
    'System Guards', 'System Guards', 'System Guards', 'System Guards',
    'Non-Sequitur', 'Non-Sequitur', 'Non-Sequitur',
][:N_TURNS]

df['phase'] = df.groupby('architecture').cumcount().map(dict(enumerate(phase_labels)))

print(f"Total turns across all variants: {len(df)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('SPEAR vs HyDE Variants: Performance Comparison', fontsize=14, fontweight='bold')

dims = {
    'Rule Accuracy': 'async_critic_scores.rule_accuracy',
    'Narrative Coherence': 'async_critic_scores.narrative_coherence',
    'NPC Voice': 'async_critic_scores.npc_voice_consistency',
}

for ax, (label, col) in zip(axes, dims.items()):
    sns.boxplot(x='architecture', y=col, data=df, order=arch_order, palette=arch_colors, ax=ax)
    ax.set_title(label, fontweight='bold')
    ax.set_ylim(-0.5, 11)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15, labelsize=9)

    # Annotate means for each of the three bars
    for i, arch in enumerate(arch_order):
        mean = df[df['architecture'] == arch][col].mean()
        ax.text(i, mean + 0.4, f'μ={mean:.2f}', ha='center', fontweight='bold', color='black')

    # Kruskal-Wallis H-test for 3-group comparison
    groups = [df[df['architecture'] == a][col].dropna() for a in arch_order]
    stat, p = stats.kruskal(*groups)
    sig = '★ SIGNIFICANT' if p < 0.05 else f'p={p:.2f}'
    ax.set_xlabel(f"Kruskal-Wallis: {sig}", color='green' if p < 0.05 else 'gray', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Targeted Impact: Rule Accuracy in High-Difficulty Turns', fontsize=14, fontweight='bold')

# Left: Rule accuracy on RAG Edge Cases only
rag_only = df[df['phase'] == 'RAG Edge Cases']
sns.barplot(x='architecture', y='async_critic_scores.rule_accuracy', 
            data=rag_only, order=arch_order, palette=arch_colors, capsize=0.1, ax=axes[0])
axes[0].set_title('Rule Accuracy (RAG Edge Cases Only)', fontweight='bold')
axes[0].set_ylim(0, 11)
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=15)

for i, arch in enumerate(arch_order):
    mean = rag_only[rag_only['architecture'] == arch]['async_critic_scores.rule_accuracy'].mean()
    axes[0].text(i, mean + 0.3, f'μ={mean:.2f}', ha='center', fontweight='bold')

# Right: Rule accuracy delta per phase (Comparing HyDEs back to standard SPEAR)
phase_means = df.groupby(['architecture', 'phase'])['async_critic_scores.rule_accuracy'].mean().unstack(0)
delta_q25 = phase_means['SPEAR+HyDE (Qwen2.5:7B)'] - phase_means['SPEAR']
delta_p35 = phase_means['SPEAR+HyDE (Phi3.5:3.8B)'] - phase_means['SPEAR']

# Plotting deltas side-by-side per phase
delta_df = pd.DataFrame({'Qwen-Delta': delta_q25, 'Phi-Delta': delta_p35})
delta_df.plot(kind='bar', ax=axes[1], color=[arch_colors['SPEAR+HyDE (Qwen2.5:7B)'], arch_colors['SPEAR+HyDE (Phi3.5:3.8B)']], edgecolor='white')
axes[1].axhline(0, color='black', linewidth=1, linestyle='--')
axes[1].set_title('Rule Accuracy Delta vs. Standard SPEAR', fontweight='bold')
axes[1].set_ylabel('Score Delta')
axes[1].legend(fontsize=8)
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('plot/cell3_hyde_performance_lift.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Cost Analysis: Qwen2.5 vs Phi3.5 HyDE Overhead', fontsize=14, fontweight='bold')

# Left: Fetch Latency
sns.boxplot(x='architecture', y='latency_breakdown.parallel_fetch_ms', data=df, 
            order=arch_order, palette=arch_colors, ax=axes[0])
axes[0].set_title('Phase 2 Parallel Fetch Latency', fontweight='bold')
axes[0].set_ylabel('ms')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=15)

# Right: Token Breakdown
token_cols = {
    'Router': 'token_breakdown.router',
    'HyDE (Rules)': 'token_breakdown.hyde_rules',
    'HyDE (Explore)': 'token_breakdown.hyde_exploration',
    'DM Main': 'token_breakdown.dm',
}

x_pos = np.arange(len(token_cols))
width = 0.25

for i, arch in enumerate(arch_order):
    subset = df[df['architecture'] == arch]
    means = [subset[col].fillna(0).mean() if col in subset.columns else 0 for col in token_cols.values()]
    axes[1].bar(x_pos + (i - 1) * width, means, width, label=arch, color=arch_colors[arch], alpha=0.8)

axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(list(token_cols.keys()))
axes[1].set_title('Mean Token Consumption', fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
fig.suptitle('The Pareto Frontier: Quality vs. Latency Tradeoff', fontsize=14, fontweight='bold')

for arch in arch_order:
    subset = df[df['architecture'] == arch]
    ax.scatter(subset['latency_ms'], subset['avg_critic_score'], 
               color=arch_colors[arch], alpha=0.6, s=100, label=arch, edgecolors='white')
    
    # Trend line calculation
    valid = subset[['latency_ms', 'avg_critic_score']].dropna()
    if len(valid) > 2:
        m, b = np.polyfit(valid['latency_ms'], valid['avg_critic_score'], 1)
        xs = np.linspace(valid['latency_ms'].min(), valid['latency_ms'].max(), 100)
        ax.plot(xs, m * xs + b, color=arch_colors[arch], linewidth=2, linestyle='--', alpha=0.5)

ax.set_xlabel('Turn Latency (ms)', fontweight='bold')
ax.set_ylabel('Composite Quality Score (0-10)', fontweight='bold')
ax.legend()
ax.grid(True, linestyle=':', alpha=0.6)
plt.savefig('plot/cell5_3way_tradeoff.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: ROI (Efficiency)
sns.barplot(x='architecture', y='score_per_1k', data=df, order=arch_order, palette=arch_colors, ax=axes[0])
axes[0].set_title('Token Efficiency (Quality per 1k Tokens)', fontweight='bold')
axes[0].set_ylabel('Quality / 1000 Tokens')
axes[0].tick_params(axis='x', rotation=15)

# Right: Cumulative Cost
for arch in arch_order:
    subset = df[df['architecture'] == arch].reset_index(drop=True)
    ax = axes[1]
    ax.plot(subset.index, subset['total_tokens'].cumsum(), color=arch_colors[arch], marker='o', label=arch)

axes[1].set_title('Cumulative Token Expenditure', fontweight='bold')
axes[1].set_xlabel('Turn Number')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
route_cols = ['Rules', 'NPC', 'Exploration']

for i, arch in enumerate(arch_order):
    # Filter and format routes
    sub_df = df[df['architecture'] == arch]
    r_data = sub_df[['routes_fired.rules_logic', 'routes_fired.npc_lore', 'routes_fired.world_exploration']].astype(float).fillna(0)
    r_data.columns = route_cols
    
    sns.heatmap(r_data.T, cmap='Greys', cbar=False, linewidths=0.1, ax=axes[i],
                xticklabels=[f"T{j+1}" for j in range(len(r_data))])
    axes[i].set_title(f'{arch} Routing', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
summary = df.groupby('architecture').agg({
    'avg_critic_score': 'mean',
    'async_critic_scores.rule_accuracy': 'mean',
    'latency_ms': 'mean',
    'total_tokens': 'mean',
    'score_per_1k': 'mean',
    'latency_breakdown.parallel_fetch_ms': 'mean'
}).round(2)

print("\n" + "="*60)
print("FINAL ARCHITECTURAL AUDIT: SPEAR vs HyDE VARIANTS")
print("="*60)
print(summary.to_string())